In [102]:
import os
import librosa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio
import tensorflow as tf
#libraries for building the model
from tensorflow.keras.layers import BatchNormalization, Conv2D,MaxPool2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping




# Converting all the audio files into the .wav format
import soundfile as sf
import noisereduce as nr
from pydub import AudioSegment # will convert the format

from tensorflow.image import resize

import subprocess

import sys
from spleeter.separator import Separator

## Spleeter Separate

In [106]:
input_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\genrenew\semiclassical\scl13.mp3"
output_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\test_output"


In [107]:
import os
import subprocess
import sys


def spleeter_conversion(input_path, output_path):
    # 1. THE FORCE: Point this to your GLOBAL FFmpeg bin folder
    global_ffmpeg_path = r"C:\ffmpeg\bin"

    # 2. Inject it at the START of the PATH so it takes priority
    os.environ["PATH"] = global_ffmpeg_path + os.pathsep + os.environ["PATH"]

    # 3. Verify it's working before running Spleeter
    import shutil
    ffmpeg_loc = shutil.which("ffmpeg")
    print(f"--- 🔍 FFmpeg is being pulled from: {ffmpeg_loc} ---")

    if "C:\\ffmpeg" not in str(ffmpeg_loc):
        print("❌ Warning: Still not using the global version!")
    else:
        print("✅ Success: Global FFmpeg detected.")

    # 4. Define the Spleeter Command

    # Using -m spleeter for better stability in Conda
    cmd = [sys.executable, "-m", "spleeter", "separate", "-p", "spleeter:4stems", "-o", output_path, input_path]

    # 5. Execute
    try:
        subprocess.run(cmd, check=True)
        print("🚀 Separation complete using Global FFmpeg!")
    except subprocess.CalledProcessError as e:
        print(f"❌ Spleeter failed. Error: {e}")

In [108]:
# will finalize the graph and it can't be changed so need to remove this process graph resources after usage
# second time run this cell , nesting violated error, but same graph by the spleeter so run , but tenserflow diff(version graph) can't run
# ffmpeg_bin = r'C:\ffmpeg\bin'
# os.environ["PATH"] = ffmpeg_bin + os.pathsep + os.environ["PATH"]
spleeter_conversion(input_path, output_path)    

--- 🔍 FFmpeg is being pulled from: C:\ffmpeg\bin\ffmpeg.EXE ---
✅ Success: Global FFmpeg detected.
🚀 Separation complete using Global FFmpeg!


### Chunking 

In [112]:
other_stem_path = r"C:\Users\Suraj\Desktop\Projects\Indian_Music_Genre_Classifier\test_output\scl13\other.wav"

In [113]:
#chunking part
def chunking(other_stem_path, sample_set_rate = 22050, target_shape = (150,150), threshold = 0.01 ):
        # --- STEP 2: Load and Chunk ---

    try:
        audio_data, sample_rate = librosa.load(other_stem_path, sr = sample_set_rate)
    except Exception as e:
        print(f"Skipping corrupted files: {other_stem_path}: {e}")
        
    #define the duration of each chunk and overlap
    chunk_duration = 4
    overlap_duration = 2
    data = []

    #convert duration to sample
    chunk_samples = int(chunk_duration*sample_rate)
    stride = int((chunk_duration-overlap_duration)*sample_rate) ## movement(4-2) * sr -> 2 second movement
    
    # sliding window logic
    # Use range with stride to ensure 'start + chunk_samples' never exceeds len(y)
    for start in range(0, len(audio_data) - chunk_samples + 1, stride):
        end = start + chunk_samples
        chunk = audio_data[start:end]

        # QUALITY CHECK: Check if chunk has actual sound (RMS energy)
        rms = np.sqrt(np.mean(chunk**2))
        if rms > threshold: 

            # melspectrogram part, this is the matrix we are getting the 
            mel_spectrogram = librosa.feature.melspectrogram(y = chunk, sr = sample_rate, n_mels = 150) #calculating spectrogram by this
            mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref= np.max)


            # 2. APPLY NORMALIZATION (Min-Max Scaling)
            # This ensures the test data matches the 0-1 range of your training data
            mel_min = mel_spectrogram.min()
            mel_max = mel_spectrogram.max()
            if mel_max - mel_min > 0:
                mel_norm = (mel_spectrogram - mel_min) / (mel_max - mel_min)
            else:
                mel_norm = mel_spectrogram # Handle silence/constant signal

            mel_spectrogram = resize(np.expand_dims(mel_norm, axis = -1), target_shape).numpy()

            # appending the data to the lists
            data.append(mel_spectrogram)
                
    return np.array(data)
    

In [114]:
X_test = chunking(other_stem_path=other_stem_path)

In [115]:
X_test.shape

(19, 150, 150, 1)

In [116]:
model = tf.keras.models.load_model("indian_music_classifier_v2.keras")
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 150, 150, 32)      320       
                                                                 
 batch_normalization (BatchN  (None, 150, 150, 32)     128       
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 148, 148, 32)      9248      
                                                                 
 batch_normalization_1 (Batc  (None, 148, 148, 32)     128       
 hNormalization)                                                 
                                                                 
 max_pooling2d (MaxPooling2D  (None, 74, 74, 32)       0         
 )                                                               
                                                        

In [117]:
#predict function
def model_prediction(X_test, classes):
    y_pred = model.predict(X_test)
    predicted_categories = np.argmax(y_pred , axis = 1)
    unique_elements, counts = np.unique(predicted_categories, return_counts = True)
    max_element = np.max(counts)
    index = unique_elements[counts == max_element][0]
    return classes[index]

In [118]:
classes = ['bollypop', 'carnatic', 'ghazal', 'semiclassical', 'sufi']

In [119]:
result = model_prediction(X_test, classes=classes)
result

1/1 [==============================] - 1s 1s/step


'semiclassical'